In [1]:
# （1）构建 MCP 客户端：一个类封装「传输 + ClientSession + 握手」
#
# 客户端原理就是固定三步：打开传输(stdio / streamable-http) -> 建 ClientSession -> initialize 握手
# 握手之后才能 list_tools（发现工具）、call_tool（调用工具）
#
# mcp SDK v2(2.3.0) 的关键变化（本 notebook 已按 v2 改写）：
#   FastMCP                 -> MCPServer                （服务端类改名）
#   streamablehttp_client   -> streamable_http_client   （旧名已删除）
#                              且不再有 headers=/timeout= 参数，改成传自己的 httpx2.AsyncClient
#   httpx                   -> httpx2                   （SDK 依赖换库）
#   result.isError          -> result.is_error          （字段统一 snake_case）
#   tool.inputSchema        -> tool.input_schema
#   InitializeResult.serverInfo -> .server_info
#   stdio_client 只 yield (read, write) 两个流，不再多返回 session_id 回调
#
# SDK 另有高层 Client，功能等价于下面整个 MCPClient，不想手写握手时直接用：
#   from mcp import Client
#   async with Client("https://xxx/mcp") as client:                    # URL = streamable-http
#       await client.list_tools(); await client.call_tool("名字", {})
#   async with Client(StdioServerParameters(command=..., args=[...])) as client: ...   # 子进程

import sys
from contextlib import AsyncExitStack, asynccontextmanager
from importlib.metadata import version
from pathlib import Path

import httpx2
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.streamable_http import streamable_http_client

# 公开可直连的远程天气 MCP 服务端：免注册免 Key，数据来自 Open-Meteo
# 工具 weather_now / air_quality_now / nearby_sensors，城市名必须用英文（Beijing）
REMOTE_WEATHER_URL = "https://modelmarket.dev/tools/mcp/weather"


class MCPClient:
    """一个 MCP 连接 = 一条传输通道 + 一个 ClientSession。"""

    def __init__(self, transport, errlog_path=None):
        self._transport = transport      # 无参调用即得到「传输的异步上下文管理器」
        self.errlog_path = errlog_path   # stdio 模式下 Server 的 stderr 写到哪个文件
        self._stack = None
        self.session = None
        self.server_info = None          # 握手拿到：服务端身份 name / version
        self.protocol_version = None     # 握手协商出的协议版本

    # ---- 两种传输：本地子进程 stdio / 远程 streamable-http ----

    @classmethod
    def stdio(cls, command, args, errlog_path=None):
        """本地 Server：当子进程拉起，用 stdin/stdout 通信。"""
        params = StdioServerParameters(command=command, args=list(args))
        # Jupyter 里 sys.stderr 是 OutStream、没有 fileno()，不能直接当子进程 stderr
        # （会抛 io.UnsupportedOperation: fileno），必须重定向到真实文件。
        # 在 .py 脚本里可省略这段，用默认的 errlog=sys.stderr 即可。
        log_path = Path(errlog_path or Path(args[0]).with_suffix(".stderr.log"))

        @asynccontextmanager
        async def transport():
            with open(log_path, "a", encoding="utf-8", errors="replace") as errlog:
                async with stdio_client(params, errlog=errlog) as streams:
                    yield streams

        return cls(transport, log_path)

    @classmethod
    def http(cls, url, headers=None, timeout=30):
        """远程 Server：streamable-http。v2 里 headers/timeout 挂在自己的 httpx2 客户端上。"""

        @asynccontextmanager
        async def transport():
            async with httpx2.AsyncClient(
                headers=headers or {},
                # 读超时给足：服务端可能长时间挂着响应流
                timeout=httpx2.Timeout(timeout, read=300),
            ) as http_client:
                async with streamable_http_client(url, http_client=http_client) as streams:
                    yield streams

        return cls(transport)

    # ---- 生命周期：进入时握手，退出时自动断开 ----

    async def __aenter__(self):
        self._stack = AsyncExitStack()
        read, write = await self._stack.enter_async_context(self._transport())
        self.session = await self._stack.enter_async_context(ClientSession(read, write))
        init = await self.session.initialize()   # 握手：协议版本 + 能力 + 服务端身份
        self.server_info = init.server_info
        self.protocol_version = init.protocol_version
        return self

    async def __aexit__(self, *exc):
        await self._stack.aclose()

    # ---- 工具发现与调用 ----

    async def list_tools(self):
        """tools/list：工具列表（name / description / input_schema / output_schema）。"""
        return (await self.session.list_tools()).tools

    async def call_tool(self, name, arguments=None):
        """tools/call：CallToolResult（content 是内容块列表，structured_content 是结构化结果）。"""
        return await self.session.call_tool(name, arguments or {})


print("MCPClient 已定义，mcp SDK 版本 =", version("mcp"))


MCPClient 已定义，mcp SDK 版本 = 2.3.0


In [2]:
# （2）构建 MCP 服务端，并用（1）的客户端连它
# Server 必须是独立进程，所以先把源码写成 .py 文件，再以子进程 + stdio 拉起
# （不能在 cell 里直接 mcp.run()：它内部调 asyncio.run，而 cell 已有运行中的事件循环）
#
# 服务端原理：@mcp.tool() 把普通函数注册成 MCP 工具
#   参数类型注解 -> input_schema（参数 JSON Schema）
#   返回类型注解 -> output_schema；返回 Pydantic 模型时，客户端直接拿到 structured_content(dict)
#   docstring    -> 工具描述（LLM 靠它判断何时调用该工具）

SERVER_DIR = Path.cwd() if Path("demo-mcp.ipynb").exists() else Path.cwd() / "demo"
SERVER_PATH = (SERVER_DIR / "weather_mcp_server.py").resolve()

SERVER_CODE = r'''"""自建天气 MCP 服务端（本地，stdio 传输）"""

import logging
import sys
from datetime import date

from pydantic import BaseModel

from mcp.server import MCPServer

# stdio 模式下 stdout 被 JSON-RPC 报文独占，任何 print() 都会破坏协议，日志一律写 stderr
# Windows 下 stdout 默认 GBK，中文会写坏报文，必须切 UTF-8
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")
logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[weather-server] %(message)s")
logger = logging.getLogger("weather-server")

# 1. 创建 Server（name / version 会在握手时告诉客户端）
mcp = MCPServer("local-weather-server", version="1.0.0")

MOCK_WEATHER = {
    "北京": ("晴", 25, 40),
    "上海": ("多云", 22, 65),
}


class Weather(BaseModel):
    """返回类型声明成 Pydantic 模型，客户端就能直接拿到 structured_content。"""

    city: str
    date: str
    condition: str
    temp_c: int
    humidity_pct: int


# 2. 注册工具
@mcp.tool()
def get_weather(city: str) -> Weather:
    """查询指定城市当天的天气信息。

    Args:
        city: 城市名称，如：北京、上海
    """
    logger.info("get_weather called: city=%s", city)
    condition, temp, humidity = MOCK_WEATHER.get(city, ("未知", 0, 0))
    return Weather(city=city, date=date.today().isoformat(), condition=condition,
                   temp_c=temp, humidity_pct=humidity)


@mcp.tool()
def list_supported_cities() -> list[str]:
    """列出当前服务支持查询的所有城市。"""
    return list(MOCK_WEATHER)


# 3. 以 stdio 方式对外服务
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

SERVER_PATH.write_text(SERVER_CODE, encoding="utf-8")
print(f"已生成服务端: {SERVER_PATH}")


async def block2():
    # 用当前 kernel 的解释器启动 Server 子进程
    async with MCPClient.stdio(sys.executable, [str(SERVER_PATH)]) as client:
        print(f"握手完成: {client.server_info.name} v{client.server_info.version}"
              f" / 协议 {client.protocol_version}")

        # tools/list：动态发现工具及其 Schema（客户端事先并不知道服务端有什么工具）
        for t in await client.list_tools():
            print(f"\n工具 {t.name}")
            print(f"  input_schema  = {t.input_schema}")
            print(f"  output_schema = {t.output_schema}")

        # tools/call
        r = await client.call_tool("get_weather", {"city": "北京"})
        print(f"\ncall_tool(get_weather) is_error = {r.is_error}")
        print(f"  structured_content = {r.structured_content}")  # 给程序读：直接是 dict
        print(f"  content[0].text    = {r.content[0].text}")     # 给 LLM 读：内容块里的文本

        cities = await client.call_tool("list_supported_cities")
        print(f"  支持城市 = {cities.structured_content}")

        # Server 端日志：logging 只能写 stderr，stdout 是协议通道
        log = Path(client.errlog_path).read_text(encoding="utf-8", errors="replace").strip()
        print(f"\nServer 日志 {Path(client.errlog_path).name}:\n  " + "\n  ".join(log.splitlines()[-3:]))


await block2()   # cell 里用顶层 await；.py 里写 asyncio.run(block2())


已生成服务端: E:\git\ai-agent-writer-dev\demo\weather_mcp_server.py


握手完成: local-weather-server v1.0.0 / 协议 2025-11-25

工具 get_weather
  input_schema  = {'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'type': 'object', 'title': 'get_weatherArguments'}
  output_schema = {'properties': {'city': {'title': 'City', 'type': 'string'}, 'date': {'title': 'Date', 'type': 'string'}, 'condition': {'title': 'Condition', 'type': 'string'}, 'temp_c': {'title': 'Temp C', 'type': 'integer'}, 'humidity_pct': {'title': 'Humidity Pct', 'type': 'integer'}}, 'required': ['city', 'date', 'condition', 'temp_c', 'humidity_pct'], 'type': 'object', 'description': '返回类型声明成 Pydantic 模型，客户端就能直接拿到 structured_content。', 'title': 'Weather'}

工具 list_supported_cities
  input_schema  = {'properties': {}, 'type': 'object', 'title': 'list_supported_citiesArguments'}
  output_schema = {'properties': {'result': {'items': {'type': 'string'}, 'title': 'Result', 'type': 'array'}}, 'required': ['result'], 'type': 'object', 'title': 'list_supported_citiesOutpu


call_tool(get_weather) is_error = False
  structured_content = {'city': '北京', 'date': '2026-10-22', 'condition': '晴', 'temp_c': 25, 'humidity_pct': 40}
  content[0].text    = {
  "city": "北京",
  "date": "2026-10-22",
  "condition": "晴",
  "temp_c": 25,
  "humidity_pct": 40
}
  支持城市 = {'result': ['北京', '上海']}

Server 日志 weather_mcp_server.stderr.log:
  [weather-server] get_weather called: city=北京


In [3]:
# （3）连接远程 MCP 服务端（streamable-http 传输）
# 与（2）的唯一区别是传输方式：MCPClient.stdio -> MCPClient.http，协议 API 完全一样
# 这正是 MCP 的价值：客户端代码不需要关心服务端在本地还是远端

async def block3():
    async with MCPClient.http(REMOTE_WEATHER_URL) as client:
        print(f"握手完成: {client.server_info.name} v{client.server_info.version}"
              f" / 协议 {client.protocol_version}")
        print("tools/list ->", [t.name for t in await client.list_tools()])

        # tools/call：按城市名查（城市名必须用英文）
        r = await client.call_tool("weather_now", {"city": "Beijing"})
        reading = r.structured_content["result"]["reading"]
        print("\nweather_now(city='Beijing') is_error =", r.is_error)
        print("  观测值 =", reading["values"])
        print("  时间/地点 =", reading["ts"], "/", r.structured_content["result"]["resolved"]["matched_place"])

        # 同一个工具改用经纬度查：中文城市名会被 GeoNames 拒绝，坐标一定可用
        by_ll = await client.call_tool("weather_now", {"latitude": 39.9042, "longitude": 116.4074})
        print("weather_now(lat/lon)  =", by_ll.structured_content["result"]["reading"]["values"])

        air = await client.call_tool("air_quality_now", {"city": "Beijing"})
        print("air_quality_now       =", air.structured_content["result"]["reading"]["values"])


await block3()   # cell 里用顶层 await；.py 里写 asyncio.run(block3())


握手完成: modelmarket-weather v1.0.0 / 协议 2025-11-25


tools/list -> ['weather_now', 'air_quality_now', 'nearby_sensors']



weather_now(city='Beijing') is_error = False
  观测值 = {'temperature_c': 15.1, 'humidity_pct': 41, 'pressure_hpa': 1016.8, 'wind_mps': 3.04}
  时间/地点 = 2026-10-22T05:20:23Z / Beijing


weather_now(lat/lon)  = {'temperature_c': 15.1, 'humidity_pct': 41, 'pressure_hpa': 1016.8, 'wind_mps': 3.04}


air_quality_now       = {'pm2_5_ugm3': 53.8, 'pm10_ugm3': 58.8, 'us_aqi': 168, 'european_aqi': 111}


In [4]:
# （4）MCP 调用的原理：不用 SDK，直接手写 JSON-RPC 报文
# 上面 ClientSession 做的全部事情，本质就是往 Server 的 stdin 写 JSON 行、从 stdout 读 JSON 行
# stdio 传输 = 一行一条 JSON-RPC 2.0 消息：
#   请求 {"jsonrpc":"2.0","id":1,"method":"...","params":{...}}  -> 服务端回同 id 的响应
#   通知 {"jsonrpc":"2.0","method":"..."}（没有 id）             -> 服务端不回应
# 注意：线上字段是 camelCase（inputSchema / isError / structuredContent），
#      而 SDK v2 的 Python 对象属性是 snake_case（input_schema / is_error / structured_content）

import json
import subprocess

server = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
    text=True, encoding="utf-8",
)


def rpc(method, params=None, msg_id=None):
    """发一条 JSON-RPC 消息。msg_id=None 表示通知（不等响应），否则读响应直到 id 匹配。"""
    payload = {"jsonrpc": "2.0", "method": method}
    if params is not None:
        payload["params"] = params
    if msg_id is not None:
        payload["id"] = msg_id
    server.stdin.write(json.dumps(payload) + "\n")
    server.stdin.flush()
    if msg_id is None:
        return None
    while True:  # 跳过中间插进来的服务端通知（没有 id 的消息）
        msg = json.loads(server.stdout.readline())
        if msg.get("id") == msg_id:
            return msg


# ① initialize：客户端报自己的身份和协议版本，服务端回能力 + 身份
print("① initialize ->", json.dumps(rpc("initialize", {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "raw-client", "version": "0.1"},
}, msg_id=1), ensure_ascii=False))

# ② initialized 通知：告诉服务端握手完成（通知没有响应，发了就继续）
rpc("notifications/initialized")
print("② notifications/initialized -> 通知，无响应")

# ③ tools/list：发现工具
print("③ tools/list ->", json.dumps(rpc("tools/list", msg_id=2), ensure_ascii=False)[:280], "...")

# ④ tools/call：调用工具
print("④ tools/call ->", json.dumps(rpc("tools/call", {
    "name": "get_weather", "arguments": {"city": "北京"}}, msg_id=3), ensure_ascii=False))

# ⑤⑥ 两类错误的区别
print("⑤ 工具不存在 ->", json.dumps(rpc("tools/call", {
    "name": "nope", "arguments": {}}, msg_id=4), ensure_ascii=False))
print("   上面是 result.isError=true：工具级错误，当成对话内容回给 LLM，让它纠正后重试")
print("⑥ 方法不存在 ->", json.dumps(rpc("bogus/method", msg_id=5), ensure_ascii=False))
print("   上面是顶层 error 对象：协议级错误，SDK 在这里会抛 MCPError 异常")

server.terminate()


① initialize -> {"jsonrpc": "2.0", "id": 1, "result": {"capabilities": {"prompts": {"listChanged": false}, "resources": {"listChanged": false, "subscribe": false}, "tools": {"listChanged": false}}, "protocolVersion": "2025-06-18", "serverInfo": {"name": "local-weather-server", "version": "1.0.0"}}}
② notifications/initialized -> 通知，无响应
③ tools/list -> {"jsonrpc": "2.0", "id": 2, "result": {"tools": [{"description": "查询指定城市当天的天气信息。\n\n    Args:\n        city: 城市名称，如：北京、上海\n    ", "inputSchema": {"properties": {"city": {"title": "City", "type": "string"}}, "required": ["city"], "type": "object", "title": "get_weatherArguments"}, ...
④ tools/call -> {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"text": "{\n  \"city\": \"北京\",\n  \"date\": \"2026-10-22\",\n  \"condition\": \"晴\",\n  \"temp_c\": 25,\n  \"humidity_pct\": 40\n}", "type": "text"}], "isError": false, "structuredContent": {"city": "北京", "date": "2026-10-22", "condition": "晴", "temp_c": 25, "humidity_pct": 40}}}
⑤ 工具不存在 -> 